In [1]:
# notebook to generate random bedtools intervals for comparing activity of random sequences for dELS predictions
# overview:
# 1. generate 100M random 371 bp intervals.
# 2. subtract ENCODE cCREs and exons from random intervals
# 3. convert to a VCF and generate predictions

In [135]:
# import packages
import pandas as pd
import os
import pybedtools
from collections import Counter

In [136]:
# generate random intervals
#!bedtools random -n 10000000 -l 371 -g /projects/tewhey-lab/buttsj/genomes/GRCh38_no_alt_analysis_set_GCA_000001405.15.genome > /projects/tewhey-lab/buttsj/Variant_Effects/revision_experiments/distal_cre_sat_mut/encode_cCRE_all/raw_data/random_intervals4dELS_comp.bed

In [137]:
# subtract enhancers and exon bed file we used for other things
# open random intervals
rand_bed = pybedtools.BedTool('../raw_data/random_intervals4dELS_comp.bed').sort()
# open encode cCREs
encode_cCREs = pybedtools.BedTool('../raw_data/GRCh38-cCREs.bed').sort()
# open exon file
exons = pybedtools.BedTool('../raw_data/gencode.v44.basic.annotation.exons.splice.autosomes.v2.093025.bed').sort()
# filter random intervals
filtered_bed = rand_bed.intersect(encode_cCREs, v=True).intersect(exons, v=True).to_dataframe()
# filter for only autosomes
filtered_autosome_bed = filtered_bed[filtered_bed['chrom'].isin([f'chr{i}' for i in range(1,23)])]
# take the center position of the interval for generating the VCF file
filtered_autosome_bed.loc[:,'center'] = [int((start + (end + 1)) / 2) for start, end in zip(filtered_autosome_bed['start'],
                                                                                            filtered_autosome_bed['end'])]

/tmp/ipykernel_3374807/2967447104.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_autosome_bed.loc[:,'center'] = [int((start + (end + 1)) / 2) for start, end in zip(filtered_autosome_bed['start'],


In [138]:
# generate a bed file of SNPs to get the reference allele at the center of the interval
filtered_snp_bed = pybedtools.BedTool.from_dataframe(pd.DataFrame({
    0 : filtered_autosome_bed['chrom'],
    1 : filtered_autosome_bed['center'] - 1,
    2 : filtered_autosome_bed['center']
}).drop_duplicates()).sort()

In [141]:
# get the reference allele
genome_fasta = '/projects/tewhey-lab/buttsj/genomes/GRCh38_no_alt_analysis_set_GCA_000001405.15.fasta'
center_ref = filtered_snp_bed.sequence(fi=genome_fasta, tab=True)

In [142]:
# iterate through reference alleles and drop intervals that have an 'N' at that position
# make empty lists for storing chrom, pos, ref to build out VCF
chrom2vcf = []
pos2vcf = []
ref2vcf = []

with open(center_ref.seqfn) as f:
    for line in f:
        chrom_coords, sequence = line.strip().split('\t')
        # check if the reference allele is an 'N' and skip if it is
        if sequence == 'N':
            continue
        else:
            # append chromosome
            chrom2vcf.append(chrom_coords.split(':')[0])
            # append position
            pos2vcf.append(int(chrom_coords.split(':')[-1].split('-')[0]) + 1)
            # append reference allele
            ref2vcf.append(sequence)
# build the datafame
vcf = pd.DataFrame({
    'chrom' : chrom2vcf,
    'pos' : pos2vcf,
    'id' : ['.' for i in range(len(chrom2vcf))],
    'ref' : ref2vcf,
    'alt' : ['A' if i == 'G' else 'T' if i == 'C' else 'G' if i == 'A' else 'C' for i in ref2vcf],
    'qual' : ['.' for i in range(len(chrom2vcf))],
    'filter' : ['.' for i in range(len(chrom2vcf))],
    'info' : ['.' for i in range(len(chrom2vcf))]
})

In [143]:
# sample 1 million random snps for generating predictions
oneM_vcf = vcf.sample(1000000).reset_index(drop=True)

In [144]:
# iterate through and save chromosome specific vcfs
for chrom in [f'chr{i}' for i in range(1,23)]:
    # filter for only that chromosome
    chrom_vcf = oneM_vcf[oneM_vcf['chrom'] == chrom]
    # save to disk
    chrom_vcf.to_csv(f'../processed_data/random_interval_vcfs/{chrom}_for_random_interval_preds.tsv', sep = '\t', index = False, header = None)

In [169]:
# get the intervals that were randomly selected for generating predictions and save as a bed file
# add id for filtering the original bed file
oneM_vcf.loc[:,'interval_id'] = [f'{chrom}_{center}' for chrom, center in zip(oneM_vcf['chrom'], oneM_vcf['pos'])]
# add id to bed file
filtered_autosome_bed.loc[:,'interval_id'] = [f'{chrom}_{center}' for chrom, center in zip(filtered_autosome_bed['chrom'], filtered_autosome_bed['center'])]
# filter the full bed file for those that randomly sampled
oneM_bed = filtered_autosome_bed[filtered_autosome_bed['interval_id'].isin(oneM_vcf['interval_id'].tolist())]
# drop duplicates
oneM_bed_noDupes = oneM_bed.filter(['chrom', 'start', 'end', 'interval_id']).drop_duplicates()
# save to disk
oneM_bed_noDupes.to_csv('../raw_data/1M_random_intervals_for_preds.bed', sep = '\t', index = False, header = None)